<a href="https://colab.research.google.com/github/Innovatewithapple/Pipelines-CrossValidation-Regularisation/blob/main/ClassificationPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [194]:
# basic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# model + preprocessing
from sklearn.model_selection import train_test_split,cross_validate,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#model
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

#Evaluation
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

In [195]:
df = pd.read_csv('/content/diabetes.csv')
df.sample(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
122,2,107,74,30,100,33.6,0.404,23,0
317,3,182,74,0,0,30.5,0.345,29,1
734,2,105,75,0,0,23.3,0.560,53,0
585,1,93,56,11,0,22.5,0.417,22,0
327,10,179,70,0,0,35.1,0.200,37,0
496,5,110,68,0,0,26.0,0.292,30,0
178,5,143,78,0,0,45.0,0.190,47,0
509,8,120,78,0,0,25.0,0.409,64,0
702,1,168,88,29,0,35.0,0.905,52,1
297,0,126,84,29,215,30.7,0.520,24,0


In [196]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [197]:
x = df.drop(columns=['Outcome','BloodPressure','SkinThickness','Insulin','DiabetesPedigreeFunction'])
y = df['Outcome']

In [198]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [203]:
#transorm
scalingColumns = ['Pregnancies','Glucose','BMI','Age']
preprocessor = ColumnTransformer([
    ('nums',StandardScaler(),scalingColumns)
])

In [204]:
#Logistic
pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model',LogisticRegression())
])

In [205]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  ['Pregnancies', 'Glucose',
                                                   'BMI', 'Age'])])),
                ('model', LogisticRegression())])

In [208]:
cross_score = cross_validate(pipeline,x_train,y_train,cv=5,scoring=["accuracy", "precision", "recall", "f1"])
cross_score

{'fit_time': array([0.03182006, 0.03711081, 0.06540155, 0.02570152, 0.02184772]),
 'score_time': array([0.05273151, 0.09692359, 0.07634616, 0.03735185, 0.04398155]),
 'test_accuracy': array([0.84259259, 0.75925926, 0.77570093, 0.73831776, 0.76635514]),
 'test_precision': array([0.83870968, 0.71428571, 0.68571429, 0.66666667, 0.70967742]),
 'test_recall': array([0.68421053, 0.52631579, 0.64864865, 0.48648649, 0.57894737]),
 'test_f1': array([0.75362319, 0.60606061, 0.66666667, 0.5625    , 0.63768116])}

In [209]:
# SVM
pipeline_svm = Pipeline([
    ('preprocess',preprocessor),
    ('model',SVC())
])

In [210]:
pipeline_svm.fit(x_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  ['Pregnancies', 'Glucose',
                                                   'BMI', 'Age'])])),
                ('model', SVC())])

In [212]:
cross_scorepipeline_svm = cross_validate(pipeline_svm,x_train,y_train,cv=5,scoring=["accuracy", "precision", "recall", "f1"])
cross_scorepipeline_svm

{'fit_time': array([0.01827717, 0.01930046, 0.0157156 , 0.018116  , 0.01638341]),
 'score_time': array([0.03073716, 0.02008462, 0.02050066, 0.02686262, 0.0214076 ]),
 'test_accuracy': array([0.82407407, 0.78703704, 0.74766355, 0.71962617, 0.76635514]),
 'test_precision': array([0.80645161, 0.74193548, 0.63888889, 0.64      , 0.70967742]),
 'test_recall': array([0.65789474, 0.60526316, 0.62162162, 0.43243243, 0.57894737]),
 'test_f1': array([0.72463768, 0.66666667, 0.63013699, 0.51612903, 0.63768116])}

In [213]:
grid_params = {
    "model__C":[0.1,1,10],
    "model__kernel":["linear","rbf"],
    "model__gamma":["scale",'auto']
}

grid = GridSearchCV(pipeline_svm,grid_params,cv=5,scoring='f1')
grid.fit(x_train,y_train)

best_estimator = grid.best_estimator_
print(best_estimator)

y_pred = best_estimator.predict(x_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))




Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  ['Pregnancies', 'Glucose',
                                                   'BMI', 'Age'])])),
                ('model', SVC(C=0.1, kernel='linear'))])
[[125  26]
 [ 35  45]]
              precision    recall  f1-score   support

           0       0.78      0.83      0.80       151
           1       0.63      0.56      0.60        80

    accuracy                           0.74       231
   macro avg       0.71      0.70      0.70       231
weighted avg       0.73      0.74      0.73       231

